# Tugas 1B: Advanced TF-IDF & Text Summarization (Manchester Football)

**Objective:** Menganalisis berita sepak bola menggunakan perhitungan TF-IDF manual dan implementasi Text Summarization.

## 1. Persiapan Data & Library

In [ ]:
import pandas as pd
import math
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    "Manchester United adalah klub bola besar di Inggris",
    "Manchester City menjuarai liga Inggris musim ini",
    "Derby Manchester selalu dinantikan oleh fans bola"
]
stopwords = set(["adalah", "di", "ini", "oleh", "selalu"])
print("Corpus Manchester loaded.")

## 2. Preprocessing & Tokenization (Regex)

In [ ]:
def preprocess(text, remove_stop=True):
    text = re.sub(r'[^\w\s]', '', text.lower())
    tokens = text.split()
    if remove_stop:
        tokens = [t for t in tokens if t not in stopwords]
    return tokens

tokenized_corpus = [preprocess(d) for d in corpus]
vocab = sorted(list(set([word for doc in tokenized_corpus for word in doc])))
print(f"Vocab size: {len(vocab)}")

## 3. Perhitungan Manual TF-IDF

In [ ]:
def get_tf_matrix(token_docs, vocabulary):
    tf_matrix = []
    for doc in token_docs:
        row = []
        total_terms = len(doc)
        for word in vocabulary:
            count = doc.count(word)
            row.append(count / total_terms if total_terms > 0 else 0)
        tf_matrix.append(row)
    return np.array(tf_matrix)

def get_idf_vector(token_docs, vocabulary):
    N = len(token_docs)
    idf_vector = []
    for word in vocabulary:
        df = sum(1 for doc in token_docs if word in doc)
        idf_vector.append(math.log10(N / df))
    return np.array(idf_vector)

tf_matrix = get_tf_matrix(tokenized_corpus, vocab)
idf_vector = get_idf_vector(tokenized_corpus, vocab)
tfidf_manual = tf_matrix * idf_vector

df_tfidf_manual = pd.DataFrame(tfidf_manual, columns=vocab, index=[f"Doc {i+1}" for i in range(len(corpus))])
display(df_tfidf_manual[vocab[:5]].head())

## 4. Analisis Kata Spesifik: 'manchester'

In [ ]:
target_word = 'manchester'
if target_word in df_tfidf_manual.columns:
    word_scores = df_tfidf_manual[target_word]
    print(f"Skor TF-IDF Manual untuk kata '{target_word}':")
    display(word_scores)
    print("\nAnalisis: Karena kata ini ada di semua dokumen, skor IDF-nya 0 (Skor Manual).")

## 5. Implementasi Text Summarization (Kalimat Terpenting)
Menggunakan rata-rata skor TF-IDF untuk menentukan kalimat yang paling merepresentasikan inti berita sepak bola.

In [ ]:
sentence_scores = []
for i in range(len(corpus)):
    scores = tfidf_manual[i]
    avg_score = np.mean(scores[scores > 0]) if any(scores > 0) else 0
    sentence_scores.append(avg_score)

df_summary = pd.DataFrame({'Original Sentence': corpus, 'Importance Score': sentence_scores}).sort_values(by='Importance Score', ascending=False)

print("TOP 3 RINGKASAN BERITA SEPAK BOLA (SUMMARIZATION):")
display(df_summary.head(3))

## 6. Perbandingan dengan Library (TfidfVectorizer)

In [ ]:
vectorizer = TfidfVectorizer(stop_words=list(stopwords))
tfidf_lib = vectorizer.fit_transform(corpus)

df_tfidf_lib = pd.DataFrame(tfidf_lib.toarray(), columns=vectorizer.get_feature_names_out(), index=[f"Doc {i+1}" for i in range(len(corpus))])
print("Hasil TF-IDF dari Scikit-Learn (Library):")
display(df_tfidf_lib[sorted(list(set(df_tfidf_lib.columns) & set(vocab)))[:5]].head())

## 7. Kesimpulan
1. **Summarization**: Berhasil menarik kalimat paling informatif dari korpus Manchester.
2. **Manual vs Library**: Scikit-Learn memberikan nilai non-nol pada kata umum karena fitur smoothing.